# AutoData — PaySim cross-schema GPU benchmark

This notebook is the **second-dataset generalization test** for AutoData.

It intentionally does **not** rewrite E0/E1/E2 for PaySim. The adapter only maps PaySim's source semantics into canonical AutoData roles:

- `isFraud` → target
- `nameOrig` → entity/source account
- `step` → hourly event-time axis (`_autodata_event_time`)
- `amount` → amount
- `type` → transaction category
- `nameDest` → counterparty / merchant-like history role

The default `strict` profile excludes balance-before/after fields and `isFlaggedFraud` to reduce simulator-specific shortcuts. An optional `full_research` ablation retains balance fields but still excludes `isFlaggedFraud`.

Primary metric: **PR-AUC**. Training may be fraud-enriched; validation/test are natural-prevalence temporal samples.


In [ ]:
# 0) Confirm Colab GPU runtime
import os, sys, json, zipfile, copy
from pathlib import Path
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Colab: Runtime > Change runtime type > T4/L4/A100")
print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi || true


In [ ]:
# 1) Locate or upload the latest AutoData project ZIP
CONTENT = Path('/content')
PROJECT_ROOT = CONTENT / 'transaction-data-intelligence'

if not (PROJECT_ROOT / 'src').exists():
    candidates = list(CONTENT.glob('AutoData*.zip')) + list(CONTENT.glob('*.zip'))
    zpath = candidates[0] if candidates else None
    if zpath is None:
        from google.colab import files
        print('Upload AutoData_v2_paysim_adapter.zip')
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No project ZIP uploaded')
        zpath = CONTENT / next(iter(uploaded))
    print('Extracting', zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(CONTENT)

if not (PROJECT_ROOT / 'src').exists():
    matches = [p.parent for p in CONTENT.rglob('config.yaml') if (p.parent / 'src').exists()]
    if len(matches) != 1:
        raise RuntimeError(f'Could not uniquely locate project root: {matches}')
    PROJECT_ROOT = matches[0]

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
# 2) Install core backend dependencies
!pip install -q -r requirements.txt
import torch
assert torch.cuda.is_available()
DEVICE='cuda'
print('GPU ready:', torch.cuda.get_device_name(0))


## Dataset setup

Download the PaySim CSV separately and place it in Google Drive. The commonly used filename is similar to `PS_20174392719_1491204439457_log.csv`.

The default source cap keeps feature generation T4/Colab-RAM friendly while preserving a continuous chronological slice. Increase only after the first run succeeds.


In [ ]:
# 3) Controls — T4-safe starting point
from google.colab import drive
drive.mount('/content/drive')

PAYSIM_PATH = '/content/drive/MyDrive/paysim/PS_20174392719_1491204439457_log.csv'
PROFILE = 'strict'                 # strict | full_research
MAX_SOURCE_ROWS = 1_000_000       # read a continuous chronological source slice; None = full 6.3M source
ROWS = 300_000                    # sampled train/validation/test rows from the source slice
SEEDS = [42, 123, 456]
EPOCHS = 10
PATIENCE = 4
BATCH_SIZE = 256
RUN_FULL_RESEARCH_ABLATION = False

print({
    'PROFILE': PROFILE, 'MAX_SOURCE_ROWS': MAX_SOURCE_ROWS, 'ROWS': ROWS,
    'SEEDS': SEEDS, 'EPOCHS': EPOCHS, 'BATCH_SIZE': BATCH_SIZE,
    'RUN_FULL_RESEARCH_ABLATION': RUN_FULL_RESEARCH_ABLATION,
})


In [ ]:
# 4) Load PaySim and apply the explicit schema/time adapter
import pandas as pd
from src.ingestion.adapters import adapt_paysim
from src.ingestion.schema_detector import detect_schema
from src.ingestion.roles import detect_roles, schema_for_profiling
from src.preprocessing.levels import infer_roles
from src.utils.config import load_config

path = Path(PAYSIM_PATH)
if not path.exists():
    raise FileNotFoundError(f'PaySim CSV not found: {path}')

raw = pd.read_csv(path, nrows=MAX_SOURCE_ROWS)
print(f'Loaded source rows: {len(raw):,}')
print('Source frauds:', int(pd.to_numeric(raw['isFraud'], errors='coerce').fillna(0).sum()))
print('Source fraud rate:', float(pd.to_numeric(raw['isFraud'], errors='coerce').mean()))

adapter = adapt_paysim(raw, profile=PROFILE)
df = adapter.df
schema = detect_schema(df, f'paysim_{PROFILE}')
roles = detect_roles(df, schema, target=adapter.target).with_overrides(
    df,
    role_overrides=adapter.role_overrides,
    entity=adapter.entity,
    datetime=adapter.datetime,
)
ps = schema_for_profiling(schema, roles, df)
level_roles = infer_roles(df, ps, roles, hints=adapter.hints)

print('Adapter profile:', adapter.profile)
print('Dropped:', adapter.dropped_columns)
print('Dataset roles:', {'target': roles.target, 'entity': roles.entity, 'datetime': roles.datetime, 'task': roles.task})
print('E2 roles:', level_roles.as_dict())
print('Notes:')
for n in adapter.notes:
    print('-', n)

assert roles.task == 'binary_classification'
assert level_roles.amount == 'amount'
assert level_roles.category == 'type'
assert level_roles.merchant == 'nameDest'


In [ ]:
# 5) Natural validation/test; enriched training only
from src.preprocessing.levels import DataPreparer

cfg = load_config()
benchmark_cfg = copy.deepcopy(cfg)
benchmark_cfg['sampling']['train_positive_share'] = 0.10
benchmark_cfg['sampling']['validation_positive_share'] = None
benchmark_cfg['sampling']['keep_all_test_positives'] = False
benchmark_cfg['features']['enabled_groups'] = ['temporal','history','sequence']

preparer = DataPreparer(df, ps, level_roles, benchmark_cfg, rows=ROWS, seed=cfg['project']['seed'])
split_info = preparer.prepare_split()
print(json.dumps(split_info['sample'], indent=2, default=str))

for split in ['train','validation','test']:
    s=split_info['sample'][split]
    print(split, 'rows=', s['rows'], 'positives=', s['positives'], 'sample prevalence=', s['sample_positive_share'])

if split_info['sample']['test']['positives'] < 50:
    print('WARNING: fewer than 50 fraud positives in test. Increase MAX_SOURCE_ROWS/ROWS before making a strong conclusion.')


In [ ]:
# 6) Build E2 once and require point-in-time safety
probe = preparer.build('E2')
pit = probe.info.get('point_in_time', {})
print(json.dumps(pit, indent=2, default=str))
assert pit.get('passed'), pit
print('PASS: PaySim E2 behavioral features use strictly earlier event steps only.')


In [ ]:
# 7) Run the strict cross-schema benchmark: E0 vs E1 continuous vs E2 full
import pandas as pd
from src.evaluation.experiment import run_comparison_multiseed, aggregate_seeds

settings={'epochs':EPOCHS,'patience':PATIENCE,'batch_size':BATCH_SIZE}

def make_preparer(config):
    p=DataPreparer(df, ps, level_roles, config, rows=ROWS, seed=cfg['project']['seed'])
    p.prepare_split()
    return p

def run_variant(name, level, config):
    print(f'\n=== {name} ===')
    p=make_preparer(config)
    if level=='E2':
        b=p.build('E2')
        assert b.info.get('point_in_time',{}).get('passed')
    out=run_comparison_multiseed(
        p,[level],config,SEEDS,settings=settings,device='cuda',
        progress=lambda done,total,seed,r: print(
            f"  seed={seed}: PR-AUC={r.metrics['test_unweighted']['pr_auc']:.6f} "
            f"F1={r.metrics['test_unweighted']['f1']:.6f}")
    )
    table=aggregate_seeds(out).reset_index().rename(columns={'Level':'level'})
    table.insert(0,'variant',name)
    rows=[]
    for seed, results in out.items():
        m=results[0].metrics['test_unweighted']
        rows.append({'seed':seed, **{k:m[k] for k in ['pr_auc','roc_auc','precision','recall','f1']}})
    u=pd.DataFrame(rows)
    for metric in ['pr_auc','roc_auc','precision','recall','f1']:
        table[f'natural_{metric} mean']=u[metric].mean()
        table[f'natural_{metric} std']=u[metric].std(ddof=1)
    return table

variants=[]
c=copy.deepcopy(benchmark_cfg)
variants.append(run_variant('E0_raw','E0',c))

c=copy.deepcopy(benchmark_cfg)
c['representation']['numeric_mode']='continuous'
c['representation']['numeric_coarse_bins']=0
variants.append(run_variant('E1_continuous','E1',c))

c=copy.deepcopy(benchmark_cfg)
c['features']['enabled_groups']=['temporal','history','sequence']
variants.append(run_variant('E2_full','E2',c))

result_table=pd.concat(variants,ignore_index=True)
display(result_table.sort_values('natural_pr_auc mean',ascending=False))


In [ ]:
# 8) Optional: quantify simulator-specific balance-field effect
# This reruns the same benchmark under full_research. It is deliberately OFF by default.
full_research_table = None
if RUN_FULL_RESEARCH_ABLATION:
    full_adapter = adapt_paysim(raw, profile='full_research')
    full_df = full_adapter.df
    full_schema = detect_schema(full_df, 'paysim_full_research')
    full_roles = detect_roles(full_df, full_schema, target=full_adapter.target).with_overrides(
        full_df, role_overrides=full_adapter.role_overrides,
        entity=full_adapter.entity, datetime=full_adapter.datetime)
    full_ps = schema_for_profiling(full_schema, full_roles, full_df)
    full_level_roles = infer_roles(full_df, full_ps, full_roles, hints=full_adapter.hints)

    def full_make(config):
        p=DataPreparer(full_df, full_ps, full_level_roles, config, rows=ROWS, seed=cfg['project']['seed'])
        p.prepare_split(); return p

    tables=[]
    for name, level, numeric_mode in [('E0_raw','E0','quantile_bin'),('E1_continuous','E1','continuous'),('E2_full','E2','quantile_bin')]:
        c=copy.deepcopy(benchmark_cfg); c['representation']['numeric_mode']=numeric_mode
        print(f'\n=== full_research::{name} ===')
        p=full_make(c)
        if level=='E2': assert p.build('E2').info.get('point_in_time',{}).get('passed')
        out=run_comparison_multiseed(p,[level],c,SEEDS,settings=settings,device='cuda')
        t=aggregate_seeds(out).reset_index().rename(columns={'Level':'level'}); t.insert(0,'variant',name); t.insert(0,'profile','full_research')
        tables.append(t)
    full_research_table=pd.concat(tables,ignore_index=True)
    display(full_research_table.sort_values('pr_auc mean',ascending=False))
else:
    print('Skipped full_research balance ablation. Set RUN_FULL_RESEARCH_ABLATION=True only after strict profile completes.')


In [ ]:
# 9) Export benchmark + adapter context
from datetime import datetime
import torch

out_dir=PROJECT_ROOT/'experiments'/'paysim_gpu'
out_dir.mkdir(parents=True,exist_ok=True)
stamp=datetime.now().strftime('%Y%m%d_%H%M%S')
csv_path=out_dir/f'paysim_{PROFILE}_{stamp}.csv'
json_path=out_dir/f'paysim_{PROFILE}_context_{stamp}.json'
result_table.to_csv(csv_path,index=False)
if full_research_table is not None:
    full_research_table.to_csv(out_dir/f'paysim_full_research_{stamp}.csv',index=False)

context={
    'dataset':'PaySim', 'source_file':path.name, 'profile':PROFILE,
    'source_rows_loaded':len(raw), 'rows_requested':ROWS,
    'source_frauds':int(pd.to_numeric(raw['isFraud'],errors='coerce').fillna(0).sum()),
    'source_fraud_rate':float(pd.to_numeric(raw['isFraud'],errors='coerce').mean()),
    'adapter':{
        'target':adapter.target,'entity':adapter.entity,'datetime':adapter.datetime,
        'hints':adapter.hints,'dropped_columns':adapter.dropped_columns,
        'notes':adapter.notes,'metadata':adapter.metadata,
    },
    'split':split_info,'point_in_time_audit':pit,
    'seeds':SEEDS,'epochs':EPOCHS,'patience':PATIENCE,'batch_size':BATCH_SIZE,
    'device':torch.cuda.get_device_name(0),
    'evaluation_distribution':'fraud-enriched train; natural temporal validation/test',
    'decision_note':'Judge cross-schema generalization from strict profile first; full_research is only a balance-field shortcut ablation.',
}
json_path.write_text(json.dumps(context,indent=2,default=str))
print(csv_path); print(json_path)

from google.colab import files
files.download(str(csv_path)); files.download(str(json_path))
